# Alpha-FX: Intelligent Forex Trading System

Welcome to **Alpha-FX**, a Deep Reinforcement Learning (DRL) system designed to trade foreign exchange markets.

## 🎯 Goal
The goal of this notebook is to walk you through the entire lifecycle of an algorithmic trading agent:
1.  **DataOps**: Ingesting and processing financial data (Prices, Indicators, Macro yields).
2.  **Training**: Teaching an AI agent (PPO) to trade profitably.
3.  **Evaluation**: Testing the agent against a "Buy & Hold" baseline to verify performance.

## 🌍 FX Context
We trade a portfolio of major currency pairs (`EURUSD`, `GBPUSD`, etc.) against the US Dollar.
*   **The Strategy**: The agent learns to allocate portfolio weights (e.g., 50% EUR, 50% USD) to maximize returns while managing risk (volatility).
*   **The Edge**: We provide the agent with **Macroeconomic Data** (US Treasury Yields) so it can understand interest rate trends (Carry Trade).

---

### 🔎 System Overview
The data flows from raw Yahoo Finance inputs to a trained model.

```mermaid
graph LR
    A[Yahoo Finance] -->|Fetch| B(Data Pipeline)
    B -->|Clean & Feature Eng| C[(Parquet Dataset)]
    C -->|Train| D[RL Agent (PPO)]
    D -->|Evaluate| E[Backtest & Benchmark]
```

## 1. Setup & Dependencies
First, we ensure all necessary libraries are installed and importable.

In [ ]:
import sys
import os
import pandas as pd
import gymnasium as gym
import stable_baselines3

# Add root to path so we can import local modules
sys.path.append(os.path.abspath('.'))

print(f"Pandas: {pd.__version__}")
print(f"Gymnasium: {gym.__version__}")
print(f"Stable Baselines3: {stable_baselines3.__version__}")

## 2. Data Operations (DataOps)

We need to fetch historical data for our currency pairs. We also fetch **Macro Data** (US 10-Year Treasury Yield `^TNX`) to help the agent understand the broader economic environment.

### Pipeline Steps:
1.  **Fetch**: Download OHLCV data from Yahoo Finance (2018-2023).
2.  **Clean**: Fix missing values (Market holidays).
3.  **Feature Engineering**: Add technical indicators:
    *   **Trend**: MACD, ADX.
    *   **Volatility**: ATR, Bollinger Bands.
    *   **Momentum**: RSI, Williams %R.
    *   **Macro**: US Risk-Free Rate (Yield).
4.  **Save**: Output to `data/fx_data_2018_2023.parquet`.

In [ ]:
from planning.run_pipeline import main as run_pipeline

# Customize Data Scope
TICKERS = ['EURUSD=X', 'GBPUSD=X', 'JPY=X', 'SEK=X', 'EURSEK=X']
START_DATE = '2018-01-01'
END_DATE = '2023-01-01'

# Run the Data Pipeline
run_pipeline(tickers=TICKERS, start_date=START_DATE, end_date=END_DATE)

In [ ]:
# Inspect the generated data
df = pd.read_parquet('data/fx_data_2018_2023.parquet')
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample Data:")
df.head()

## 3. Training the Agent

We use **PPO (Proximal Policy Optimization)**, a state-of-the-art reinforcement learning algorithm. 
*   **configuration**: We use hyperparameters tuned via Optuna (Batch=256, Steps=2048) to maximize stability.

You can now control the **Observation Duration** (Training steps). A longer duration usually leads to better performance, but takes longer.

In [ ]:
from train_agent import train

# Interactive Training Control
# Short run for demo: 10,000 steps
# High Performance run: 100,000+ steps
TOTAL_TIMESTEPS = 30000 

print(f"Starting training for {TOTAL_TIMESTEPS} steps...")
train(total_timesteps=TOTAL_TIMESTEPS)

## 4. Evaluation & Benchmarking

Does the agent actually make money? We compare it against a **Benchmark**: 
*   **Strategy**: Buy & Hold (Equal Weights).
*   **Dataset**: 2022 Validation Set (Unseen during training).

We look at:
1.  **Cumulative Return**: Total profit %.
2.  **Sharpe Ratio**: Risk-adjusted return (Higher is better).
3.  **Max Drawdown**: Maximum peak-to-trough drop (Risk).

In [ ]:
from benchmark_agent import run_benchmark

# Run Benchmark Analysis
run_benchmark()

## 5. Conclusion

You have successfully ran the Alpha-FX pipeline.

*   **Next Steps**: 
    *   Try increasing `TOTAL_TIMESTEPS` to `100000` to see if the agent learns a better policy.
    *   Add more tickers (e.g., `'AUDUSD=X'`) in the DataOps section.